# Object detection 1: build data_object/

A detector learns from **boxes**, and our photos only have a class per photo. This notebook gives the photos boxes and writes them as a detection dataset, `data_object/` (YOLO format), used by notebooks 02 (YOLO) and 03 (our MobileNetV3 + Faster R-CNN).

1. **Choose the photos:** up to `MAX_TRAIN_PER_CLASS` per class from `data/<class>/train` (a validation share is held out to pick the best epoch) and `TEST_PER_CLASS` from `data/<class>/test`.
2. **Human boxes from Open Images** for the photos that came from there (all banana, orange and mushroom photos, some pine cones, sunflowers and dogs). Google's box files are streamed once and only our photos' rows are kept.
3. **OWLv2 boxes** for every other photo (pseudo-labels: the ready-made detector from the API). Its results are cached, so an interrupted run continues where it stopped.
4. **Write `data_object/`** and contact sheets to spot-check the boxes.

**On Kaggle** (recommended: OWLv2 needs a GPU for thousands of photos): attach the dataset made from `imgcls_data.zip` (as for the classifier notebooks), turn on a GPU and *Internet*, run all, then *Save Version*. In the saved version's *Output*, make a new dataset from the `data_object` folder and name it `data_object`; attach that dataset to notebooks 02 and 03.
**Locally**: `data_object/` is written in the project folder (git-ignored). On this laptop's CPU OWLv2 takes about 8 s per photo, so use `LIMIT_PER_CLASS` for a try, or run it on the GPU laptop.

Generated by `scripts/create_detection_notebooks.py` on 2026-11-24. To reset this notebook: `python scripts/create_detection_notebooks.py --only 01 --force`

In [ ]:
# ---- settings for this model (edit freely) ----
MAX_TRAIN_PER_CLASS = 700       # photos per class from data/<class>/train (butterfly has 6,500); None = all
TEST_PER_CLASS = 150            # photos per class from data/<class>/test; None = all
VAL_FRACTION = 0.1              # share of the train photos held out to pick the best epoch
USE_OPENIMAGES_BOXES = True     # human boxes where they exist (streams about 2.4 GB of box files once)
DETECTOR = 'google/owlv2-base-patch16-ensemble'  # labels the photos that have no human boxes
PSEUDO_THRESHOLD = 0.3          # OWLv2 boxes scoring below this are not used as labels
MAX_SIDE = 640                  # images are saved with their longer side at most this many pixels
DEVICE = 'auto'                 # "auto" = GPU if available; OWLv2 takes about 8 s per photo on a CPU, under 0.5 s on a GPU
LIMIT_PER_CLASS = None          # e.g. 5 for a quick smoke test (overrides the two caps above)
OBJECT_DIR = None               # None = data_object/ in the project (Kaggle: /kaggle/working/data_object)
MAKE_ZIP = False                # also write data_object.zip next to the folder, to upload as a Kaggle dataset
SEED = 42

## Setup
Finds the classifier photos (`data/` locally, the attached dataset on Kaggle) and the device.

In [ ]:
import subprocess
import sys
from pathlib import Path

# find the project (the folder holding imgcls/); on Colab / Kaggle fetch it from GitHub first
REPO_URL = "https://github.com/M0HiT27/Image-Classification.git"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imgcls").is_dir()), None)
if ROOT is None:
    on_colab = Path("/content").is_dir() and not Path("/kaggle").is_dir()  # Kaggle has /content too
    ROOT = Path("/content" if on_colab else "/tmp") / "Image-Classification"
    if (ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(ROOT)], check=True)
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))
# forget an imgcls loaded earlier in this session, so re-running this cell after a pull uses the new code
for name in [m for m in sys.modules if m == "imgcls" or m.startswith("imgcls.")]:
    del sys.modules[name]


def ensure(module, pip_name=None):
    """Install a package that is missing (Kaggle and Colab have most of them already)."""
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True)


ensure('timm')
ensure('transformers')
ensure('cv2', 'opencv-python-headless')

from IPython.display import Image, display
import imgcls
from imgcls import objdet

from imgcls.objdet import build

env = imgcls.setup_environment(colab_drive_dir="/content/drive/MyDrive/imgcls")
imgcls.set_seed(SEED)
device = imgcls.get_device(DEVICE)
object_dir = Path(OBJECT_DIR).expanduser().resolve() if OBJECT_DIR else objdet.default_object_dir(env)
cache_dir = object_dir / "_cache"  # Open Images rows and OWLv2 results, reused by later runs
print(f"data_object will be written to {object_dir}")

## 1. Choose the photos
Photos per class and split; *from Open Images* photos may have human boxes.

In [ ]:
photos = build.select_photos(MAX_TRAIN_PER_CLASS, TEST_PER_CLASS, VAL_FRACTION, LIMIT_PER_CLASS, SEED)
display(build.photo_table(photos))

## 2. Human boxes from Open Images
Looks up the boxes of **all** our Open Images photos once (not only the chosen ones), so later builds with other settings need no download. The train-split box file is 2.26 GB: on Kaggle that takes about a minute, on a 10 MB/s connection about 4 minutes. Nothing but our rows is saved.

In [ ]:
human = build.openimages_boxes(cache_dir) if USE_OPENIMAGES_BOXES else {}
need_owl = [p for p in photos if not build.human_boxes(p, human)]
per_photo = 8.0 if device.type == "cpu" else 0.4  # rough seconds per photo for OWLv2
print(f"{len(photos) - len(need_owl)} of {len(photos)} photos have human boxes; OWLv2 will look at the "
      f"other {len(need_owl)} (roughly {len(need_owl) * per_photo / 60:.0f} min on this device, less if "
      "some are cached)")

## 3. OWLv2 boxes for the other photos
Each photo keeps OWLv2's boxes of its own class (any dog box for the four breeds: the breed comes from the folder). Results are appended to `_cache/` as they come, so stopping this cell loses nothing: run it again to continue.

In [ ]:
owl = build.owlv2_boxes(need_owl, cache_dir, device=device, model_id=DETECTOR)

## 4. Write data_object/
Photos where neither source found a box of their class are left out (*no box* columns): without boxes they would teach the detector that the object is background.

In [ ]:
table = build.write_dataset(photos, human, owl, object_dir, threshold=PSEUDO_THRESHOLD, max_side=MAX_SIDE,
                            settings=imgcls.config_from_globals(globals()))
display(table)

## Spot-check the boxes
One row per class: green boxes were drawn by people (Open Images), orange ones by OWLv2. Look for missing objects, boxes around the wrong thing and boxes around only a part. The test boxes are what both detectors are scored against, so they matter most.

In [ ]:
for split in ("train", "test"):
    print(split)
    display(Image(filename=str(build.contact_sheet(object_dir, split))))

## Upload to Kaggle (optional)
On Kaggle, make the dataset straight from the notebook's output folder (see the top). Locally, `MAKE_ZIP = True` writes `data_object.zip` next to the folder for *New Dataset* > upload.

In [ ]:
if MAKE_ZIP:
    build.make_zip(object_dir, object_dir.parent / "data_object.zip")